In [1]:
import numpy as np
import pandas as pd
import os
import joblib
from glob import glob

In [2]:
df = pd.read_csv("../data/final_data.csv")

TARGET = "P_HABITABLE"

FEATURES = [col for col in df.columns if col != TARGET]

X_all_unscaled = df[FEATURES].values
y_true = df[TARGET].values

In [3]:
model_files = sorted(glob("../results/margin_run*_final.joblib"))

probs_all = []

for f in model_files:
    artifact = joblib.load(f)
    clf = artifact["clf"]
    scaler = artifact["scaler"]

    X_scaled = scaler.transform(X_all_unscaled)
    probs = clf.predict_proba(X_scaled)[:, 1]
    probs_all.append(probs)

probs_all = np.vstack(probs_all)

In [4]:
df["prob_mean"] = probs_all.mean(axis=0)
df["prob_std"]  = probs_all.std(axis=0)

In [5]:
# Only unlabeled planets
candidates = df[df[TARGET] == 0].copy()

# Sort by mean probability
candidates = candidates.sort_values("prob_mean", ascending=False)

In [6]:
candidates.head(5)[["prob_mean", "prob_std"]]

,prob_mean,prob_std
5272,0.816244,0.059721
2595,0.293450,0.045848
2335,0.292776,0.045661
208,0.129312,0.016242
5271,0.091815,0.019613


In [7]:
recommended = candidates.iloc[0]

recommendation_table = recommended[
    FEATURES + ["prob_mean", "prob_std"]
].to_frame(name="Value")

recommendation_table

,Value
pl_orbper,636.130000
pl_orbsmax,1.334000
pl_orbeccen,0.160000
P_RADIUS,1.810000
P_MASS,3.930000
P_DENSITY,0.662761
P_FLUX,0.282046
P_TEMP_EQUIL,184.744770
st_teff,5310.000000
st_mass,0.783000


In [8]:
habitables = df[df[TARGET] == 1]

context_df = pd.DataFrame({
    "Candidate": recommended[FEATURES],
    "Habitable_Median": habitables[FEATURES].median(),
    "Habitable_IQR": habitables[FEATURES].quantile(0.75) - habitables[FEATURES].quantile(0.25)
})

In [9]:
context_df

,Candidate,Habitable_Median,Habitable_IQR
pl_orbper,636.130000,36.769980,92.304175
pl_orbsmax,1.334000,0.164700,0.289517
pl_orbeccen,0.160000,0.083470,0.140993
P_RADIUS,1.810000,1.780000,0.896000
P_MASS,3.930000,4.060000,3.477500
P_DENSITY,0.662761,0.681304,0.464153
P_FLUX,0.282046,0.851427,0.676961
P_TEMP_EQUIL,184.744770,244.031560,47.354197
st_teff,5310.000000,3460.000000,919.500000
st_mass,0.783000,0.415000,0.364250


In [10]:
import shap

artifact = joblib.load(model_files[0])
clf = artifact["clf"]
scaler = artifact["scaler"]

X_scaled = scaler.transform(X_all_unscaled)

explainer = shap.TreeExplainer(clf)

In [11]:
idx = recommended.name
shap_values = explainer.shap_values(X_scaled[idx:idx+1])

In [12]:
shap_df = pd.DataFrame({
    "feature": FEATURES,
    "shap_value": shap_values[0],
    "feature_value": X_all_unscaled[idx]
}).sort_values("shap_value", ascending=False)

shap_df

,feature,shap_value,feature_value
6,P_FLUX,1.283584,0.282046
3,P_RADIUS,1.012418,1.810000
7,P_TEMP_EQUIL,0.625044,184.744770
12,sy_dist,0.480447,3.603040
2,pl_orbeccen,0.145053,0.160000
0,pl_orbper,0.129792,636.130000
5,P_DENSITY,0.106744,0.662761
4,P_MASS,0.089332,3.930000
13,sy_pnum,0.077159,4.000000
11,S_LUMINOSITY,0.076863,0.495450


In [21]:
candidate_params = {
    "pl_orbper": recommendation_table.loc['pl_orbper'].loc['Value'],
    "pl_orbsmax": recommendation_table.loc['pl_orbsmax'].loc['Value'],
    "P_RADIUS": recommendation_table.loc['P_RADIUS'].loc['Value'],
    "P_MASS": recommendation_table.loc['P_MASS'].loc['Value'],
    "st_teff": recommendation_table.loc['st_teff'].loc['Value']
}
candidate_params

{'pl_orbper': np.float64(636.13),
 'pl_orbsmax': np.float64(1.334),
 'P_RADIUS': np.float64(1.81),
 'P_MASS': np.float64(3.93),
 'st_teff': np.float64(5310.0)}

In [22]:
# --- Load datasets ---
nasa = pd.read_csv("../data/PSCompPars_2025.10.22_08.26.30.csv", header=323)
phl = pd.read_csv("../data/hwc_2025.10.22.csv")

In [23]:
# --- Normalize planet names for merging ---
nasa["pl_name_norm"] = nasa["pl_name"].str.strip().str.lower()
phl["p_name_norm"] = phl["P_NAME"].str.strip().str.lower()

In [24]:
# --- Merge datasets ---
merged = pd.merge(
    nasa,
    phl,
    left_on="pl_name_norm",
    right_on="p_name_norm",
    how="inner"
)

print(f"Merged planets: {len(merged)}")
print("Merged columns:", len(merged.columns))
merged.head()

Merged planets: 5576
Merged columns: 440


,rowid,pl_name,hostname,pl_letter,hd_name,hip_name,tic_id,gaia_dr2_id,gaia_dr3_id,sy_snum,...,S_TIDAL_LOCK,P_HABZONE_OPT,P_HABZONE_CON,P_TYPE_TEMP,P_HABITABLE,P_ESI,S_CONSTELLATION,S_CONSTELLATION_ABR,S_CONSTELLATION_ENG,p_name_norm
0,1,11 Com b,11 Com,b,HD 107383,HIP 60202,TIC 72437047,Gaia DR2 3946945413106333696,Gaia DR3 3946945413106333696,2,...,0.589839,0,0,Hot,0,0.087644,Coma Berenices,Com,Berenice's Hair,11 com b
1,2,11 UMi b,11 UMi,b,HD 136726,HIP 74793,TIC 230061010,Gaia DR2 1696798367260229376,Gaia DR3 1696798367260229376,1,...,0.541702,0,0,Hot,0,0.081366,Ursa Minor,UMi,Little Bear,11 umi b
2,3,14 And b,14 And,b,HD 221345,HIP 116076,TIC 333225860,Gaia DR2 1920113512486282240,Gaia DR3 1920113512486282240,1,...,0.557058,0,0,Hot,0,0.077422,Andromeda,And,Andromeda,14 and b
3,4,14 Her b,14 Her,b,HD 145675,HIP 79248,TIC 219483057,Gaia DR2 1385293808145621504,Gaia DR3 1385293808145621504,1,...,0.434927,0,0,Cold,0,0.163020,Hercules,Her,Hercules,14 her b
4,5,16 Cyg B b,16 Cyg B,b,HD 186427,HIP 96901,TIC 27533327,Gaia DR2 2135550755683407232,Gaia DR3 2135550755683407232,3,...,0.512355,1,1,Warm,0,0.368093,Cygnus,Cyg,Swan,16 cyg b b


In [25]:
tol = {
    "pl_orbper": 1e-3,
    "pl_orbsmax": 1e-4,
    "P_RADIUS": 1e-3,
    "P_MASS": 1e-2,
    "st_teff": 5.0
}

In [26]:
mask = (
    np.isclose(merged["pl_orbper"], candidate_params["pl_orbper"], atol=tol["pl_orbper"]) &
    np.isclose(merged["pl_orbsmax"], candidate_params["pl_orbsmax"], atol=tol["pl_orbsmax"]) &
    np.isclose(merged["P_RADIUS"], candidate_params["P_RADIUS"], atol=tol["P_RADIUS"]) &
    np.isclose(merged["P_MASS"], candidate_params["P_MASS"], atol=tol["P_MASS"]) &
    np.isclose(merged["st_teff"], candidate_params["st_teff"], atol=tol["st_teff"])
)

matches = merged[mask]
matches

,rowid,pl_name,hostname,pl_letter,hd_name,hip_name,tic_id,gaia_dr2_id,gaia_dr3_id,sy_snum,...,S_TIDAL_LOCK,P_HABZONE_OPT,P_HABZONE_CON,P_TYPE_TEMP,P_HABITABLE,P_ESI,S_CONSTELLATION,S_CONSTELLATION_ABR,S_CONSTELLATION_ENG,p_name_norm
5567,6020,tau Cet f,tau Cet,f,HD 10700,HIP 8102,TIC 419015728,Gaia DR2 2452378776434276992,Gaia DR3 2452378776434477184,1,...,0.499084,0,0,Cold,0,0.554636,Cetus,Cet,Whale,tau cet f


In [27]:
matched_row = matches.iloc[0]

matched_row[
    [
        "pl_name",
        "hostname",
        "discoverymethod",
        "disc_year",
        "sy_dist",
        "sy_pnum",
        "pl_orbper",
        "pl_orbsmax",
        "P_RADIUS",
        "P_MASS",
        "st_teff",
        "st_rad",
        "st_lum"
    ]
]

pl_name                  tau Cet f
hostname                   tau Cet
discoverymethod    Radial Velocity
disc_year                     2017
sy_dist                    3.60304
sy_pnum                          4
pl_orbper                   636.13
pl_orbsmax                   1.334
P_RADIUS                      1.81
P_MASS                        3.93
st_teff                     5310.0
st_rad                        0.83
st_lum                    -0.30539
Name: 5567, dtype: object

In [34]:
for i in matched_row.index:
    print(f"{i}: {matched_row[i]}")

rowid: 6020
pl_name: tau Cet f
hostname: tau Cet
pl_letter: f
hd_name: HD 10700
hip_name: HIP 8102
tic_id: TIC 419015728
gaia_dr2_id: Gaia DR2 2452378776434276992
gaia_dr3_id: Gaia DR3 2452378776434477184
sy_snum: 1
sy_pnum: 4
sy_mnum: 0
cb_flag: 0
discoverymethod: Radial Velocity
disc_year: 2017
disc_refname: <a refstr=FENG_ET_AL__2017 href=https://ui.adsabs.harvard.edu/abs/2017AJ....154..135F/abstract target=ref>Feng et al. 2017</a>
disc_pubdate: 2017-10
disc_locale: Ground
disc_facility: Multiple Observatories
disc_telescope: Multiple Telescopes
disc_instrument: Multiple Instruments
rv_flag: 1
pul_flag: 0
ptv_flag: 0
tran_flag: 0
ast_flag: 0
obm_flag: 0
micro_flag: 0
etv_flag: 0
ima_flag: 0
dkin_flag: 0
pl_controv_flag: 0
pl_orbper: 636.13
pl_orbpererr1: 11.7
pl_orbpererr2: -47.69
pl_orbperlim: 0.0
pl_orbper_reflink: <a refstr=FENG_ET_AL__2017 href=https://ui.adsabs.harvard.edu/abs/2017AJ....154..135F/abstract target=ref>Feng et al. 2017</a>
pl_orbsmax: 1.334
pl_orbsmaxerr1: 0.017
p